# MultiLexNorm 2026 — Train one language on Colab

This notebook reproduces the per-language training pipeline (MFR baseline → ByT5 → mT5 → picker) on a free Colab GPU.

**Before you run:**
1. Runtime → Change runtime type → **GPU** (T4 is free; A100 if you have Pro).
2. Accept dataset access at https://huggingface.co/datasets/weerayut/multilexnorm2026-dev-pub (click *Agree and access*).
3. Get a HF read token at https://huggingface.co/settings/tokens — paste it when prompted in cell 3.

Repo: https://github.com/Ethan5767/intro-to-ai-assignment

## 1. Verify GPU and clone repo

In [ ]:
!nvidia-smi | head -n 15

In [ ]:
# Repo to clone. Change this if you have your own fork.
REPO_URL = "https://github.com/Ethan5767/intro-to-ai-assignment.git"
REPO_DIR = "intro-to-ai-assignment"

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
%cd $REPO_DIR

## 2. Install dependencies
Colab already ships with `torch` and `transformers`; this just tops up missing pieces and pins compatible versions.

In [ ]:
!pip -q install -r requirements.txt

## 3. Authenticate to Hugging Face
Datasets are gated. Paste your **read** token (starts with `hf_...`) when prompted.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Smoke-check: dataset loads and has the splits we expect
from datasets import load_dataset
ds = load_dataset("weerayut/multilexnorm2026-dev-pub")
print({k: len(v) for k, v in ds.items()})
print("Sample:", ds["train"][0])

## 4. Run unit tests
Should pass in a few seconds; this verifies the data prep, prediction reassembly, and picker logic survive any environment differences.

In [ ]:
!python -m pytest tests/ -q

## 5. MFR baseline — all 17 languages
Fast (CPU-only), no GPU needed. Writes per-language predictions and `outputs/summary_mfr.csv` (consumed by the picker).

In [ ]:
!python run_mfr.py

## 6. Train ByT5-small on one language
Pick a language code and tweak epochs / training cap. English `en` finishes in ~10–15 min on a free T4 with `--limit-train 5000 --epochs 3`.

All checkpoints land under `./ckpts/<model>/<lang>/best/` (gitignored).

In [ ]:
LANG = "en"            # one of: en de nl es it hr sr sl da tr id iden trde ja ko th vi
EPOCHS = 3
LIMIT_TRAIN = 5000     # set to -1 for the full split (slower)

!python train_seq2seq.py --model byt5-small --lang $LANG --epochs $EPOCHS --limit-train $LIMIT_TRAIN

## 7. Predict + score on dev


In [ ]:
import os
os.makedirs(f"outputs/byt5/{LANG}", exist_ok=True)

!python predict_seq2seq.py --ckpt ckpts/byt5-small/$LANG/best --lang $LANG --split validation --out-json outputs/byt5/$LANG/predictions_dev.json --batch-size 16 --num-beams 1

In [ ]:
import json
from datasets import load_dataset
from utils import evaluate

ds = load_dataset("weerayut/multilexnorm2026-dev-pub")
items = [dict(x) for x in ds["validation"] if x["lang"] == LANG]
raw = [it["raw"] for it in items]
gold = [it["norm"] for it in items]
with open(f"outputs/byt5/{LANG}/predictions_dev.json", encoding="utf-8") as f:
    preds = [r["pred"] for r in json.load(f)]
evaluate(raw, gold, preds)

## 8. (Optional) Train mT5-small on the same language for the ablation

In [ ]:
!python train_seq2seq.py --model mt5-small --lang $LANG --epochs $EPOCHS --limit-train $LIMIT_TRAIN

## 9. (Optional) Run the full driver for a few languages
`run_pipeline.py` skips any language whose checkpoint already exists, so re-running is safe.

In [ ]:
!python run_pipeline.py --model byt5-small --langs en de nl --epochs 3 --limit-train 5000

## 10. Save artefacts to Drive (optional)
Colab wipes its disk between sessions — copy the checkpoint and predictions out if you want to resume later.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/multilexnorm2026
!cp -r ckpts /content/drive/MyDrive/multilexnorm2026/ 2>/dev/null || true
!cp -r outputs /content/drive/MyDrive/multilexnorm2026/